In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 3
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
14.638034807057238

Trial 1 =========================================
13.913913576821104

Trial 2 =========================================
15.397024300269011

Trial 3 =========================================
15.355795095496482

Trial 4 =========================================
15.379461579963866

Trial 5 =========================================
14.265749865353314

Trial 6 =========================================
16.610088901840975

Trial 7 =========================================
14.44682969924896

Trial 8 =========================================
15.366206696232084

Trial 9 =========================================
18.6380196766554

Trial 10 =========================================
14.322251079917708

Trial 11 =========================================
16.957085063629552

Trial 12 =========================================
14.293829692415237

Trial 13 =========================================
13.889937558907341

Trial 14 ==========

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 15 =========================================
15.90117957612588

Trial 16 =========================================
14.772995319270038

Trial 17 =========================================
15.962728805828853

Trial 18 =========================================
14.790452388943768

Trial 19 =========================================
15.875624669702901

Trial 20 =========================================
14.620433355271485

Trial 21 =========================================
15.392920087891294

Trial 22 =========================================
14.460658545372473

Trial 23 =========================================
15.943655077401317

Trial 24 =========================================
15.39908355109846

Trial 25 =========================================
14.462232459133563

Trial 26 =========================================
13.91567778850365

Trial 27 =========================================
18.814925140505913

Trial 28 =========================================
15.375270602103537

Trial 29 

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 55 =========================================
13.428155810653347

Trial 56 =========================================
15.690529507088355

Trial 57 =========================================
14.385624305012009

Trial 58 =========================================
14.354537863859687

Trial 59 =========================================
14.44705308943584

Trial 60 =========================================
14.638207180966814

Trial 61 =========================================
16.081576104300822

Trial 62 =========================================
14.370624908272562

Trial 63 =========================================
14.495019769007438

Trial 64 =========================================
13.444424196214603

Trial 65 =========================================
15.23407064561994

Trial 66 =========================================
16.010835927249516

Trial 67 =========================================
15.3699420350874

Trial 68 =========================================
16.29016453371179

Trial 69 ==

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 84 =========================================
16.24335813214591

Trial 85 =========================================
15.533942467350526

Trial 86 =========================================
14.271427484697098

Trial 87 =========================================
15.398185495272857

Trial 88 =========================================
15.175823092878796

Trial 89 =========================================
14.635306696327852

Trial 90 =========================================
14.49871859590151



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 91 =========================================
16.005729485614136

Trial 92 =========================================
13.798328910887196

Trial 93 =========================================
14.440437976051138

Trial 94 =========================================
15.362587431699064

Trial 95 =========================================
18.801292063453158

Trial 96 =========================================
14.446062685191134

Trial 97 =========================================
16.27272176430067

Trial 98 =========================================
14.289062730280161

Trial 99 =========================================
16.007544365606744

[14.63803481 13.91391358 15.3970243  15.3557951  15.37946158 14.26574987
 16.6100889  14.4468297  15.3662067  18.63801968 14.32225108 16.95708506
 14.29382969 13.88993756 16.18783705 15.90117958 14.77299532 15.96272881
 14.79045239 15.87562467 14.62043336 15.39292009 14.46065855 15.94365508
 15.39908355 14.46223246 13.91567779 18.81492514 15.3752706  15.929222

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.814925140505913
Avg = 15.273039138376339
Std = 1.2749318072910953


In [6]:
print(y_max_arr.tolist())

[14.638034807057238, 13.913913576821104, 15.397024300269011, 15.355795095496482, 15.379461579963866, 14.265749865353314, 16.610088901840975, 14.44682969924896, 15.366206696232084, 18.6380196766554, 14.322251079917708, 16.957085063629552, 14.293829692415237, 13.889937558907341, 16.187837046861123, 15.90117957612588, 14.772995319270038, 15.962728805828853, 14.790452388943768, 15.875624669702901, 14.620433355271485, 15.392920087891294, 14.460658545372473, 15.943655077401317, 15.39908355109846, 14.462232459133563, 13.91567778850365, 18.814925140505913, 15.375270602103537, 15.929222010399386, 16.23940123079065, 14.785292778966342, 13.143066680636728, 15.351116996825164, 14.506116826504151, 13.42993899408083, 15.999401591853895, 16.039885953314034, 15.398671310795526, 15.929222010399386, 15.859829455521982, 13.020368050126788, 13.979042797723638, 13.713924138855077, 14.366915287436848, 15.398346478044953, 13.551604106992276, 15.929222010399386, 16.096202887145843, 17.401295536259262, 15.0746

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)